# Focus - Feature Engineering

This notebook will be focused on creating asset-level features from teh cleaned price and returns from the yfinacne data. THe features will be used for predictive modeling, to rpedict future realized volatility of the assets. 

# Imports

In [3]:
import pandas as pd
import numpy as np 
from pathlib import Path

# Paths Enums

In [4]:
PROCESSED_DATA_PATH = Path("../data/processed")
FEATURE_DATA_PATH = PROCESSED_DATA_PATH / "features"

FEATURE_DATA_PATH.mkdir(parents=True, exist_ok=True)

# Load Data

In [7]:
adj_close = pd.read_csv(
    PROCESSED_DATA_PATH / "adjusted_close_clean.csv",
    index_col=0,
    parse_dates=True
)

daily_returns = pd.read_csv(
    PROCESSED_DATA_PATH / "daily_returns_clean.csv",
    index_col=0,
    parse_dates=True
)

## Validate Loaded Data

In [8]:
adj_close.head()

,AAPL,AGG,AMZN,CAT,GLD,JPM,KO,LLY,LMT,MSFT,...,PG,QQQ,SPY,TLT,UNH,V,VNQ,VXUS,VZ,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-02,40.267075,85.561783,59.450500,131.293884,125.150002,85.901276,35.024227,74.897774,254.261215,78.699905,...,72.280708,150.057236,235.954300,98.705627,191.993912,107.888237,59.645279,44.600658,33.712952,58.185497
2018-01-03,40.260063,85.569626,60.209999,131.494553,124.820000,85.988815,34.947304,75.304634,256.392456,79.066170,...,72.192986,151.515305,237.446747,99.177605,194.007919,108.962318,59.472107,44.911564,33.020184,59.328266
2018-01-04,40.447071,85.514771,60.479500,133.300446,125.459999,87.220642,35.439533,75.640739,260.790466,79.762070,...,72.703331,151.780426,238.447525,99.161896,194.850037,109.367455,58.447601,45.284664,33.127247,59.410378
2018-01-05,40.907574,85.459908,61.457001,135.407318,125.330002,86.660713,35.431839,76.569427,263.153290,80.750969,...,72.751160,153.304764,240.036575,98.878716,198.565567,111.986679,58.476479,45.587799,33.051662,59.362473
2018-01-08,40.755634,85.436386,62.343498,138.810043,125.309998,86.788712,35.378010,76.180275,261.940094,80.833374,...,73.133881,153.901230,240.475525,98.815811,195.119141,112.438934,58.779476,45.564495,32.994995,59.629349


In [9]:
daily_returns.head()

,AAPL,AGG,AMZN,CAT,GLD,JPM,KO,LLY,LMT,MSFT,...,PG,QQQ,SPY,TLT,UNH,V,VNQ,VXUS,VZ,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-03,-0.000174,0.000092,0.012775,0.001528,-0.002637,0.001019,-0.002196,0.005432,0.008382,0.004654,...,-0.001214,0.009717,0.006325,0.004782,0.010490,0.009956,-0.002903,0.006971,-0.020549,0.019640
2018-01-04,0.004645,-0.000641,0.004476,0.013734,0.005127,0.014325,0.014085,0.004463,0.017153,0.008801,...,0.007069,0.001750,0.004215,-0.000158,0.004341,0.003718,-0.017227,0.008307,0.003242,0.001384
2018-01-05,0.011385,-0.000642,0.016163,0.015805,-0.001036,-0.006420,-0.000217,0.012278,0.009060,0.012398,...,0.000658,0.010043,0.006664,-0.002856,0.019069,0.023949,0.000494,0.006694,-0.002282,-0.000806
2018-01-08,-0.003714,-0.000275,0.014425,0.025130,-0.000160,0.001477,-0.001519,-0.005082,-0.004610,0.001020,...,0.005261,0.003891,0.001829,-0.000636,-0.017357,0.004038,0.005182,-0.000511,-0.001715,0.004496
2018-01-09,-0.000115,-0.002752,0.004676,0.002409,-0.004628,0.005069,0.005000,-0.000813,0.007160,-0.000679,...,-0.007305,0.000061,0.002263,-0.013373,0.004983,-0.001927,-0.012888,0.000853,-0.003668,-0.004246


# Separate out Benchmark

In [10]:
BENCHMARK = "SPY"

portfolio_assets = [ticker for ticker in daily_returns.columns if ticker != BENCHMARK]

portfolio_returns = daily_returns[portfolio_assets]
benchmark_returns = daily_returns[BENCHMARK]

portfolio_prices = adj_close[portfolio_assets]
benchmark_prices = adj_close[BENCHMARK]

Like previous notebook SPY is separated as its the benchmark. The engineered features are created for the portfolio asset, while SPY is used for benchmark-related comparision features

# Rolling Return Features

In [11]:
rolling_return_5d = portfolio_returns.rolling(window=5).mean()
rolling_return_21d = portfolio_returns.rolling(window=21).mean()
rolling_return_63d = portfolio_returns.rolling(window=63).mean()

In [16]:
rolling_return_5d.tail(5)

,AAPL,AGG,AMZN,CAT,GLD,JPM,KO,LLY,LMT,MSFT,NEE,PG,QQQ,TLT,UNH,V,VNQ,VXUS,VZ,XOM
Date,,,,,,,,,,,,,,,,,,,,
2025-12-23,-0.001620,0.000068,0.008522,-0.001956,0.008850,0.006537,-0.001418,0.003323,0.002347,0.004374,-0.003746,-0.002738,0.003693,-0.000085,-0.005684,0.004755,-0.000184,0.003900,-0.004078,0.008189
2025-12-24,0.001463,0.000568,0.009888,0.007687,0.006305,0.008886,-0.001583,0.006690,0.004625,0.004968,0.000441,-0.004489,0.007986,0.001308,-0.002434,0.006157,0.000391,0.005384,-0.002712,0.003093
2025-12-26,0.000906,0.000109,0.005046,0.006024,0.009001,0.009378,-0.001391,0.003936,0.005464,0.001540,-0.001053,-0.001045,0.005075,-0.000307,0.002266,0.005149,0.001668,0.004392,0.000381,0.004390
2025-12-29,0.000081,0.000641,0.004139,0.000846,0.000070,0.004151,0.000292,0.001363,0.006185,0.000488,0.001839,0.000177,0.001500,0.001191,0.000963,0.003057,0.002837,0.002511,0.003301,0.006517
2025-12-30,0.001558,0.000641,0.003585,-0.001725,-0.004401,0.000239,-0.000393,0.000611,0.001847,0.001055,0.001230,0.001908,0.000077,0.001148,0.004301,0.000872,0.002214,0.001783,0.003584,0.004778


In [15]:
rolling_return_21d.tail(5)

,AAPL,AGG,AMZN,CAT,GLD,JPM,KO,LLY,LMT,MSFT,NEE,PG,QQQ,TLT,UNH,V,VNQ,VXUS,VZ,XOM
Date,,,,,,,,,,,,,,,,,,,,
2025-12-23,0.000187,0.000010,0.002491,0.002889,0.004799,0.004406,-0.001688,0.000674,0.002654,0.001535,-0.002079,-0.002414,0.002630,-0.000710,0.001164,0.003663,-0.000299,0.002570,-0.001470,0.001019
2025-12-24,-0.000337,0.000021,0.001334,0.002206,0.003847,0.004883,-0.001289,0.000442,0.003975,0.001460,-0.002113,-0.000735,0.001551,-0.000693,0.001709,0.003709,-0.000114,0.002406,0.000185,0.001391
2025-12-26,-0.000589,-0.000046,0.000649,0.001547,0.004418,0.003902,-0.001465,-0.001294,0.003565,0.001129,-0.002476,-0.001142,0.001255,-0.000971,0.001247,0.002934,-0.000500,0.002115,-0.000076,0.001946
2025-12-29,-0.000626,-0.000064,0.000663,0.000590,0.001965,0.002568,-0.001445,-0.001011,0.003957,0.000220,-0.002957,-0.001121,0.000603,-0.001003,0.000332,0.002987,-0.000614,0.001513,-0.000416,0.002406
2025-12-30,-0.000968,-0.000017,-0.000086,0.000321,0.001408,0.001678,-0.001662,0.000279,0.003484,-0.000381,-0.003220,-0.001263,0.000107,-0.000891,0.000789,0.002761,-0.000675,0.001397,-0.000425,0.002110


In [17]:
rolling_return_63d.tail(5)

,AAPL,AGG,AMZN,CAT,GLD,JPM,KO,LLY,LMT,MSFT,NEE,PG,QQQ,TLT,UNH,V,VNQ,VXUS,VZ,XOM
Date,,,,,,,,,,,,,,,,,,,,
2025-12-23,0.001292,0.000155,0.001049,0.003726,0.003068,0.000798,0.000962,0.006084,0.000076,-0.000640,0.001431,-0.000823,0.000762,-0.000023,-0.001025,0.000782,-0.000221,0.000880,-0.000884,0.000866
2025-12-24,0.001090,0.000213,0.001214,0.003968,0.002936,0.000954,0.001129,0.006746,0.000244,-0.000505,0.001386,-0.000649,0.000877,0.000074,-0.000607,0.001036,-0.000069,0.000984,-0.000787,0.000696
2025-12-26,0.001153,0.000205,0.001105,0.003877,0.003030,0.000761,0.001149,0.006537,0.000053,-0.000654,0.001123,-0.000658,0.000811,0.000036,-0.000333,0.000915,-0.000208,0.000978,-0.000819,0.000458
2025-12-29,0.001238,0.000194,0.000902,0.003558,0.002077,0.000578,0.001126,0.006508,0.000098,-0.000771,0.001020,-0.000784,0.000660,-0.000035,-0.000522,0.000766,-0.000181,0.000842,-0.000688,0.001053
2025-12-30,0.001186,0.000189,0.001119,0.003338,0.001953,0.000575,0.001038,0.005726,-0.000164,-0.000862,0.001221,-0.000854,0.000580,-0.000027,-0.000372,0.000665,-0.000214,0.000804,-0.000859,0.001318


Rolling return features measure recent average performance over short, medium, and longer windows. These help capture recent momentum or weakness in each asset.

# Rolling Volatility Features

In [18]:
rolling_volatility_5d = portfolio_returns.rolling(window=5).std()
rolling_volatility_21d = portfolio_returns.rolling(window=21).std()
rolling_volatility_63d = portfolio_returns.rolling(window=63).std()

In [19]:
# Saw online it was good to have annualized one good to have as optional
TRADING_DAYS = 252

rolling_volatility_21d_annualized = rolling_volatility_21d * np.sqrt(TRADING_DAYS)

In [21]:
rolling_volatility_5d.tail(5)

,AAPL,AGG,AMZN,CAT,GLD,JPM,KO,LLY,LMT,MSFT,NEE,PG,QQQ,TLT,UNH,V,VNQ,VXUS,VZ,XOM
Date,,,,,,,,,,,,,,,,,,,,
2025-12-23,0.007802,0.001466,0.012033,0.025449,0.009938,0.010379,0.004289,0.011482,0.011873,0.007303,0.010615,0.013584,0.013231,0.003350,0.003939,0.004468,0.004546,0.005962,0.008526,0.011833
2025-12-24,0.006567,0.001716,0.010258,0.007273,0.011524,0.009291,0.004022,0.007794,0.011245,0.006911,0.010343,0.010464,0.005350,0.004249,0.007212,0.002453,0.005282,0.002738,0.010337,0.008421
2025-12-26,0.006703,0.001335,0.006466,0.008338,0.010700,0.008299,0.003875,0.006708,0.009987,0.002760,0.009681,0.008606,0.004864,0.004131,0.008210,0.003861,0.004246,0.002365,0.009314,0.006727
2025-12-29,0.006244,0.001054,0.007174,0.006624,0.026251,0.012340,0.004134,0.003848,0.010382,0.002580,0.005100,0.007903,0.004052,0.003933,0.009509,0.003841,0.002968,0.003820,0.004474,0.007168
2025-12-30,0.003629,0.001054,0.007222,0.003649,0.023062,0.009387,0.004033,0.003369,0.007281,0.002158,0.004593,0.004863,0.003850,0.003979,0.008972,0.003301,0.002677,0.003402,0.004574,0.006359


In [22]:
rolling_volatility_21d.tail(5)

,AAPL,AGG,AMZN,CAT,GLD,JPM,KO,LLY,LMT,MSFT,NEE,PG,QQQ,TLT,UNH,V,VNQ,VXUS,VZ,XOM
Date,,,,,,,,,,,,,,,,,,,,
2025-12-23,0.008501,0.001979,0.012932,0.020155,0.007216,0.016722,0.008177,0.017252,0.013908,0.012173,0.012232,0.013696,0.009955,0.004905,0.016686,0.015080,0.005164,0.005255,0.011772,0.012654
2025-12-24,0.007765,0.001990,0.011826,0.019907,0.007001,0.016731,0.008215,0.017154,0.012813,0.012162,0.012201,0.012777,0.008460,0.004929,0.016734,0.015083,0.005343,0.005254,0.010735,0.012443
2025-12-26,0.007709,0.001958,0.011405,0.019777,0.007132,0.016602,0.008220,0.014955,0.012982,0.012118,0.012024,0.012542,0.008399,0.004902,0.016254,0.014842,0.004908,0.005018,0.010564,0.012042
2025-12-29,0.007698,0.001944,0.011402,0.019703,0.012603,0.016763,0.008234,0.014937,0.013115,0.011502,0.011768,0.012542,0.008311,0.004867,0.016248,0.014825,0.004816,0.004846,0.010434,0.012237
2025-12-30,0.007609,0.001922,0.010722,0.019699,0.012375,0.016413,0.008162,0.013783,0.013136,0.011101,0.011553,0.012553,0.008150,0.004803,0.016378,0.014877,0.004772,0.004802,0.010428,0.012119


In [23]:
rolling_volatility_63d.tail(5)

,AAPL,AGG,AMZN,CAT,GLD,JPM,KO,LLY,LMT,MSFT,NEE,PG,QQQ,TLT,UNH,V,VNQ,VXUS,VZ,XOM
Date,,,,,,,,,,,,,,,,,,,,
2025-12-23,0.011252,0.001901,0.020893,0.023416,0.014575,0.014157,0.010540,0.020612,0.012532,0.012040,0.013589,0.010931,0.011323,0.005148,0.016661,0.012475,0.007712,0.007286,0.013557,0.011716
2025-12-24,0.011058,0.001907,0.020850,0.023320,0.014602,0.014203,0.010494,0.019871,0.012548,0.012025,0.013561,0.011002,0.011307,0.005205,0.016564,0.012393,0.007751,0.007236,0.013614,0.011673
2025-12-26,0.011031,0.001906,0.020835,0.023329,0.014640,0.014184,0.010484,0.019864,0.012544,0.011967,0.013431,0.011000,0.011300,0.005221,0.016645,0.012368,0.007648,0.007234,0.013600,0.011548
2025-12-29,0.011011,0.001898,0.020800,0.023346,0.015666,0.014283,0.010475,0.019871,0.012582,0.011935,0.013428,0.010959,0.011311,0.005138,0.016672,0.012335,0.007653,0.007230,0.013567,0.011143
2025-12-30,0.011021,0.001900,0.020738,0.023333,0.015645,0.014283,0.010472,0.019076,0.012444,0.011901,0.013363,0.010963,0.011314,0.005134,0.016722,0.012337,0.007639,0.007217,0.013418,0.011004


In [24]:
rolling_volatility_21d_annualized.tail(5)

,AAPL,AGG,AMZN,CAT,GLD,JPM,KO,LLY,LMT,MSFT,NEE,PG,QQQ,TLT,UNH,V,VNQ,VXUS,VZ,XOM
Date,,,,,,,,,,,,,,,,,,,,
2025-12-23,0.134950,0.031414,0.205291,0.319956,0.114550,0.265452,0.129812,0.273868,0.220777,0.193234,0.194178,0.217419,0.158030,0.077871,0.264885,0.239388,0.081972,0.083427,0.186881,0.200878
2025-12-24,0.123260,0.031591,0.187735,0.316011,0.111132,0.265591,0.130410,0.272319,0.203406,0.193059,0.193682,0.202823,0.134292,0.078252,0.265648,0.239429,0.084819,0.083409,0.170413,0.197529
2025-12-26,0.122380,0.031084,0.181054,0.313944,0.113217,0.263547,0.130481,0.237407,0.206089,0.192359,0.190882,0.199106,0.133326,0.077815,0.258032,0.235610,0.077919,0.079659,0.167694,0.191156
2025-12-29,0.122195,0.030856,0.180999,0.312773,0.200074,0.266106,0.130705,0.237118,0.208199,0.182581,0.186814,0.199098,0.131938,0.077260,0.257927,0.235335,0.076447,0.076935,0.165632,0.194261
2025-12-30,0.120785,0.030510,0.170204,0.312714,0.196439,0.260550,0.129572,0.218803,0.208521,0.176217,0.183392,0.199270,0.129385,0.076246,0.259995,0.236170,0.075750,0.076234,0.165545,0.192377


Rolling volatility measures how much each asset has been moving. Since our project will maybe focus on risk, volatility features will probably be important for future modeling.

# Moving Average Features

In [25]:
moving_average_21d = portfolio_prices.rolling(window=21).mean()
moving_average_63d = portfolio_prices.rolling(window=63).mean()

In [26]:
moving_average_21d.tail(5)

,AAPL,AGG,AMZN,CAT,GLD,JPM,KO,LLY,LMT,MSFT,NEE,PG,QQQ,TLT,UNH,V,VNQ,VXUS,VZ,XOM
Date,,,,,,,,,,,,,,,,,,,,
2025-12-23,276.729669,98.191519,228.655238,583.104417,391.890953,310.022090,69.815572,1042.525705,458.008741,480.695991,81.199435,142.483552,615.209929,86.479866,324.351179,336.212542,87.153529,73.525332,39.325026,115.462432
2025-12-24,276.629379,98.193364,228.945714,584.250029,393.401905,311.492188,69.722827,1042.849365,459.801248,481.360648,81.021954,142.366698,616.138628,86.418548,324.852057,337.437948,87.142415,73.700588,39.330098,115.615132
2025-12-26,276.459692,98.188701,229.081429,585.027204,395.147619,312.667040,69.617872,1041.321696,461.402694,481.868863,80.814423,142.190710,616.888448,86.333008,325.215195,338.408690,87.097558,73.855589,39.325026,115.831263
2025-12-29,276.279553,98.182214,229.220000,585.258603,395.884763,313.426849,69.513944,1040.106294,463.197241,481.944717,80.566982,142.018009,617.239543,86.244336,325.282475,339.396031,87.042595,73.966673,39.306580,116.101896
2025-12-30,276.005300,98.180368,229.187143,585.335894,396.409049,313.914523,69.394588,1040.309419,464.778031,481.729959,80.296535,141.825127,617.285947,86.165795,325.498242,340.305599,86.982154,74.069514,39.287674,116.340109


In [27]:
moving_average_63d.tail(5)

,AAPL,AGG,AMZN,CAT,GLD,JPM,KO,LLY,LMT,MSFT,NEE,PG,QQQ,TLT,UNH,V,VNQ,VXUS,VZ,XOM
Date,,,,,,,,,,,,,,,,,,,,
2025-12-23,266.622616,98.013948,227.964603,545.764786,378.072223,305.594653,68.188459,928.460313,470.524288,500.161194,81.046064,145.566266,609.837519,87.159462,333.606854,337.750245,87.508980,72.798083,39.341419,113.496852
2025-12-24,266.894948,98.034548,228.190476,547.683121,379.138572,305.865634,68.260358,934.209658,470.603657,499.875754,81.144698,145.462966,610.331016,87.164657,333.361410,338.080183,87.500338,72.867972,39.306105,113.569401
2025-12-26,267.183101,98.054382,228.392699,549.557292,380.249683,306.075972,68.333319,939.814014,470.592094,499.515540,81.224050,145.358141,610.785290,87.166563,333.205551,338.369413,87.479524,72.937620,39.269321,113.614920
2025-12-29,267.493262,98.073161,228.549842,551.269876,380.982064,306.226698,68.405068,945.402753,470.601460,499.096159,81.295615,145.234655,611.148091,87.162402,332.987318,338.608462,87.461048,72.997434,39.237978,113.729255
2025-12-30,267.789482,98.091470,228.755556,552.875862,381.671270,306.376304,68.471051,950.431229,470.484616,498.629943,81.382442,145.101171,611.462149,87.158961,332.817648,338.812615,87.439667,73.054656,39.199438,113.873618


In [30]:
price_to_ma_21d = portfolio_prices / moving_average_21d
price_to_ma_63d = portfolio_prices / moving_average_63d

In [31]:
price_to_ma_21d.tail(5)

,AAPL,AGG,AMZN,CAT,GLD,JPM,KO,LLY,LMT,MSFT,NEE,PG,QQQ,TLT,UNH,V,VNQ,VXUS,VZ,XOM
Date,,,,,,,,,,,,,,,,,,,,
2025-12-23,0.982384,0.999793,1.015240,0.994601,1.055498,1.041262,0.987569,1.024450,1.041243,1.008312,0.968891,0.990346,1.008834,0.993015,0.988016,1.046728,0.992330,1.021872,0.983032,1.020501
2025-12-24,0.987972,1.001978,1.015000,0.994935,1.047097,1.046650,0.992280,1.029236,1.044061,1.009340,0.979045,1.000228,1.010260,0.999739,0.994936,1.048122,0.998984,1.020252,0.992754,1.017446
2025-12-26,0.987099,1.002226,1.015010,0.992320,1.054644,1.038726,0.990374,1.031483,1.034612,1.007635,0.981071,1.003198,1.008967,0.997433,1.006719,1.044703,1.000625,1.021342,0.996822,1.014611
2025-12-29,0.989043,1.003495,1.012433,0.984458,1.006859,1.023062,0.995971,1.033627,1.043064,1.006216,0.982371,1.003239,1.003512,1.002213,0.997744,1.040519,1.003285,1.016582,0.997290,1.024314
2025-12-30,0.987566,1.003113,1.014586,0.982252,1.006259,1.020432,0.996404,1.034402,1.037666,1.007450,0.988872,1.000989,1.001109,1.000735,1.006844,1.034841,1.006012,1.017184,1.003193,1.026117


In [32]:
price_to_ma_63d.tail(5)

,AAPL,AGG,AMZN,CAT,GLD,JPM,KO,LLY,LMT,MSFT,NEE,PG,QQQ,TLT,UNH,V,VNQ,VXUS,VZ,XOM
Date,,,,,,,,,,,,,,,,,,,,
2025-12-23,1.019624,1.001604,1.018316,1.062649,1.094077,1.056348,1.011135,1.150309,1.013546,0.969071,0.970724,0.969373,1.017721,0.985273,0.960604,1.041963,0.988299,1.032081,0.982623,1.038174
2025-12-24,1.024007,1.003601,1.018360,1.061363,1.086489,1.065904,1.013539,1.148926,1.020096,0.971954,0.977564,0.978937,1.019873,0.991181,0.969539,1.046131,0.994898,1.031910,0.993360,1.035774
2025-12-26,1.021371,1.003599,1.018071,1.056367,1.095964,1.061094,1.008991,1.142891,1.014408,0.972037,0.976124,0.981338,1.019049,0.987894,0.982577,1.044824,0.996256,1.034197,0.998236,1.034403
2025-12-29,1.021530,1.004610,1.015402,1.045155,1.046243,1.047117,1.012116,1.137168,1.026652,0.971637,0.973566,0.981019,1.013514,0.991657,0.974658,1.042940,0.998485,1.030080,0.999034,1.045683
2025-12-30,1.017865,1.004022,1.016500,1.039922,1.045114,1.045539,1.009844,1.132221,1.025080,0.973304,0.975678,0.978389,1.010644,0.989332,0.984701,1.039401,1.000748,1.031315,1.005451,1.048343


Moving average features compare the current price to its recent avg price. A value above 1 means the asset is trading above its moving avg, while a val < 1 means it's trading below its moving average.